# 🛍️ Olist E-Commerce Analytics — Part 1: Data Cleaning & Wrangling
**Author:** Data Analytics Portfolio
**Dataset:** Brazilian E-Commerce Public Dataset by Olist (100k Orders, 2016–2018)
**Objective:** End-to-end data ingestion, schema validation, missing value imputation, geolocation deduplication, and master analytics dataset construction.

---
### Relational Schema Overview
The dataset contains 9 interconnected relational tables:
1. `olist_orders_dataset`: Core order tracking, timestamps, and fulfillment status.
2. `olist_customers_dataset`: Customer identifiers and location.
3. `olist_order_items_dataset`: Line-item details, pricing, and freight per seller.
4. `olist_products_dataset`: Product dimensions and categories.
5. `olist_sellers_dataset`: Seller locations.
6. `olist_order_payments_dataset`: Payment methods, installment structures, and values.
7. `olist_order_reviews_dataset`: Customer CSAT review scores and feedback text.
8. `olist_geolocation_dataset`: Brazilian zip code to lat/lng mapping (1M+ coordinates).
9. `product_category_name_translation`: Portuguese to English category mappings.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Set plotting styles
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Environment initialized successfully.")


## 1. Data Ingestion & Schema Profiling
We load all 9 raw datasets, inspect row counts, data types, and check for missing values.


In [ ]:
from src.data_loader import load_all_raw_datasets

raw_data = load_all_raw_datasets()

for name, df in raw_data.items():
    print(f"Dataset: {name:<25} | Rows: {df.shape[0]:>8,} | Columns: {df.shape[1]:>2}")


## 2. Missing Value Audit & Anomaly Detection
We analyze nulls across each dataset to determine appropriate imputation strategies.


In [ ]:
null_summary = []
for name, df in raw_data.items():
    nulls = df.isnull().sum()
    null_cols = nulls[nulls > 0]
    for col, count in null_cols.items():
        pct = (count / len(df)) * 100
        null_summary.append({"Dataset": name, "Column": col, "Missing Rows": count, "Missing %": pct})

null_df = pd.DataFrame(null_summary)
null_df.sort_values(by="Missing %", ascending=False)


## 3. Geolocation Aggregation & Deduplication
The raw geolocation table has **1,000,163 rows** with multiple jittered GPS coordinates per zip code prefix.
We calculate the median coordinates per zip code prefix to compress this to **19,015 clean centroids**, achieving a **98% reduction** in memory footprint while preserving geographic precision.


In [ ]:
from src.data_cleaning import clean_geolocation

geo_clean = clean_geolocation(raw_data['geolocation'])
print(f"Raw Geolocation Rows: {len(raw_data['geolocation']):,}")
print(f"Clean Centroid Rows:  {len(geo_clean):,}")
geo_clean.head()


## 4. Product Category Translation & Missing Value Imputation
We map Portuguese category names to English and impute unmapped categories like `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`.


In [ ]:
from src.data_cleaning import clean_category_translations

products_clean = clean_category_translations(raw_data['products'], raw_data['category_translation'])
print(f"Total Products: {len(products_clean):,}")
print(f"Unique English Categories: {products_clean['product_category_name_english'].nunique()}")
products_clean['product_category_name_english'].value_counts().head(10)


## 5. Master Analytics Dataset Construction
We execute relational merges across orders, customers, items, products, sellers, payments, and reviews, engineering logistics latency metrics (`delivery_days`, `is_late_delivery`, `freight_ratio`).


In [ ]:
from src.data_cleaning import build_master_dataset

master = build_master_dataset(raw_data)
print("Master Analytics Dataset Shape:", master.shape)
print("
Sample Columns:")
print(master[['order_id', 'customer_unique_id', 'product_category_name_english', 'price', 'freight_value', 'delivery_days', 'review_score']].head())
